# Notebook 3 - Inspecting, Preparing and Visualizing Observations

**Version:** 1.0 | **Last updated:** 2026-07-24 | 

**Author:** Eshanta Mishra | **Author institution:** EarthScope Consortium

**Estimated Time:**  ~ 30 minutes | **Pathway:**  MVP1

**License:** CC-BY-4.0

## Introduction

**What this notebook does:** It retrieves GNSS observations for several stations using the EarthScope SDK, then works through the checks you should run before using the data such as how complete it is, how it is sampled, where it is missing, and which values look wrong.

**Why it is useful:** Raw GNSS observations are never uniformly complete. Satellites rise and set, receivers go offline, and some signals are never tracked. Positions, displacements, and reflector heights are all computed from these observations, so gaps and bad values carry through into the result. Checking coverage and data quality first tells you what the later numbers can support.

**What you will accomplish:** By the end you will have mesured the time coverage and sampling interval of a real multi-station dataset, located its gaps, quantified per-satellite availability, aggregated the observations into time bins, flagged questionable signal-to-noise values without discarding them, and produced first-look plots that let you tell normal data from unusual data.

---

### Prerequisites

* [ ] Have completed [Notebook 1 - Accessing GNSS Observations with the EarthScope SDK](NB1-access-gnss-via-SDK-checkpoint.ipynb])
* [ ] Be familiar with basic python and Polar dataframes.

---

## GeoLab Compute Resources

| Setting | Recommended |
|---|---|
| Image | GeoLab (default image) |
| Server size | 4 GB RAM, ~0.5 CPUs (default server) |

## Learning Objectives

By the end of this notebook, you will be able to:

1. Inspect a retrieved observation dataframe to establish its time coverage, sampling interval, gaps, and per-satellite availability.
2. Prepare observations by accounting for missing values and aggregating them into time bins.
3. Flag implausible and low-quality SNR values without discarding them.
4. Visualize SNR over time and data availability, and compare across stations and satellites.

## Relevant Documentation & Resources

* [EarthScope SDK - GNSS Observations tutorial](https://docs.earthscope.org/sdk/gnss-obs-tutorial)
* [EarthScope SDK - GNSS Satellite Ephemeris Positions tutorial](https://docs.earthscope.org/sdk/gnss-eph-pos-tutorial)
* [GeoLab Documentation](https://docs.earthscope.org/geolab)
* [Polars User Guide](https://docs.pola.rs/)
* [Altair](https://altair-viz.github.io/)

## Contents

1. [Setup & Imports](#id-1-setup-imports)
2. [Retrieve a Working Dataset](#id-2-retrieve-a-working-dataset)
3. [Inspecting the Data](#id-3-inspecting-the-data)
4. [Preparing the Data](#id-4-preparing-the-data)
5. [Flagging Outliers](#id-5-flagging-outliers)
6. [Visualizing Observations](#id-6-visualizing-observations)
7. [Comparing Across Stations and Satellites](#id-7-comparing-across-stations-and-satellites)
8. [Exploration Exercises](#id-8-exploration-exercises)
9. [Troubleshooting & Support](#id-9-troubleshooting-support)

## 1. Setup & Imports

In [ ]:
# Standard library imports
import datetime as dt

# Third-party imports
import altair as alt
import polars as pl
from earthscope_sdk import AsyncEarthScopeClient

# Enable the Rust (vegafusion) backend so Altair can handle larger datasets efficiently
alt.data_transformers.enable("vegafusion")

es = AsyncEarthScopeClient()

### Configuration

Set your parameters here before running the rest of the notebook. Every subsequent cell reads from these variables, so this is the only place you need to edit to point the notebook at
different data.

The three stations below all belong to the permanent Alaska network. Two of them record continuously across the whole window whereas the third does not. Section 3 uses that difference to show what incomplete coverage looks like.

In [ ]:
# Modify these values before running the notebook.
STATIONS   = ["CLGO", "SELD", "PCHL"]        # three GNSS stations (4-character IDs)
SESSION    = "A"                             # session name
SYSTEM     = "G"                             # constellation: GPS
OBS_CODES  = ["1C", "2W"]                    # L1 C/A and L2 P(Y) via Z-tracking
FIELDS     = ["snr", "phase"]                # measurement columns to return

START      = dt.datetime(2025, 7, 20)        # query start (UTC)
END        = dt.datetime(2025, 7, 24)        # query end (UTC)

NOMINAL_DT = dt.timedelta(seconds=15)        # expected spacing between epochs
TIME_BIN   = "5m"                            # aggregation window used in Section 4

## 2. Retrieve a Working Dataset

**What:** Four days of GPS observations from three Alaska stations, restricted to two signals and two measurement fields, returned as an Apache Arrow table and converted to a Polars dataframe. This is the same `gnss_observations()` call you met in Notebook 1, with the filters doing the work of keeping the request small. Each row is one satellite, one signal, one epoch.

**Why:** Every filter we pass is pushed to the server, so only the data we asked for crosses the network. Dropping the constellation and observation-code filters would return several times as many rows for the same window, and requesting every field would widen each one. Filtering keeps the fetch to a minute or two.

**Expected result:** On the order of one million rows across seven columns. The fetch typically takes a minute or two.

In [ ]:
# Describe the request, then .fetch() to run the query.
table = await es.data.gnss_observations(
    start_datetime=START,
    end_datetime=END,
    station_name=STATIONS,
    session_name=SESSION,
    system=SYSTEM,
    obs_code=OBS_CODES,
    field=FIELDS,
).fetch()

df = pl.from_arrow(table).sort("timestamp")
print(f"{len(df):,} rows x {df.width} columns")
df.head()

> **Check:** You should see a dataframe with `timestamp`, `satellite`, `obs_code`, `snr`, `phase`, `system`, and `igs` columns.

## 3. Inspecting the data

These checks compare what the request returned against what it asked for. The five steps below go from the coarsest view of the dataset to the finest.

### 3.1 Structure and contents

In [ ]:
print(df.schema)
print()
print("Stations:  ", df["igs"].unique().sort().to_list())
print("Systems:   ", df["system"].unique().sort().to_list())
print("Obs codes: ", df["obs_code"].unique().sort().to_list())
print("Satellites:", sorted(df["satellite"].unique().to_list()))

**What to look for:** The stations, systems, and observation codes should match exactly what you requested in the Configuration cell. If a station you asked for is missing from this list, it returned no data at all for your window, which is a different problem from returning incomplete data.

## 3.2 Time coverage per station

A dataframe can look healthy in aggregate while one station contributes almost nothing. Splitting the coverage per station is the first place that shows up.

We compare the number of distinct epochs each station reported against the number we would expect if it had recorded continuously at the nominal interval.

In [ ]:
expected_epochs = int((END - START) / NOMINAL_DT)
print(f"Expected epochs per station over this window: {expected_epochs:,}")

coverage = (
    df.group_by("igs")
      .agg(
          pl.col("timestamp").min().alias("first_epoch"),
          pl.col("timestamp").max().alias("last_epoch"),
          pl.col("timestamp").n_unique().alias("n_epochs"),
          pl.len().alias("n_rows"),
      )
      .with_columns(
          (pl.col("n_epochs") / expected_epochs * 100).round(1).alias("completeness_pct")
      )
      .sort("igs")
)
coverage

> **Check:** Two of the three stations should sit close to 100% completeness and start at `2025-07-20 00:00:00`. The third should show a noticeably later `first_epoch` and a completeness figure well under 50%. That station was not reporting for the earlier part of the window. This is real, and it is exactly the kind of thing that would quietly bias a result computed across all three stations without checking.

### 3.3 Sampling Interval

Notebook 2 found the sampling interval of the instantaneous position stream by taking the difference between consecutive timestamps. That worked because each position stream has exactly
one row per epoch.

Observations are different. A single epoch contains one row per satellite, per signal, so a single timestamp is repeated many times over. Differencing the raw `timestamp` column
therefore returns mostly zeros and tells you nothing about the sampling rate. The cell below shows this.

In [ ]:
# This does NOT give the sampling interval, because many rows share the same epoch.
(
    df.select(pl.col("timestamp").diff().alias("dt"))["dt"]
      .drop_nulls()
      .value_counts()
      .sort("count", descending=True)
      .head()
)

The fix is to reduce to the set of distinct epochs first, for one station at a time. Stations are independent recorders, so mixing them together would interleave their epochs and
give meaningless differences.

In [ ]:
def epoch_spacing(frame: pl.DataFrame, station_igs: str) -> pl.DataFrame:
    """Distribution of time gaps between consecutive distinct epochs at one station."""
    return (
        frame.filter(pl.col("igs") == station_igs)
             .select("timestamp")
             .unique()
             .sort("timestamp")
             .select(pl.col("timestamp").diff().alias("dt"))["dt"]
             .drop_nulls()
             .value_counts()
             .sort("count", descending=True)
    )


for station_igs in df["igs"].unique().sort():
    print(f"--- {station_igs} ---")
    print(epoch_spacing(df, station_igs).head())

**What to look for:** For this window each station returns exactly one spacing, `15s`, which is the nominal sampling interval of these receivers. A single-row result is the healthy case.

On other windows you may see more rows. A few large values scattered among the 15-second majority are ordinary interruptions, and Section 3.4 locates them.

### 3.4 Locating the gaps

Whether gaps exist, and where, is a separate question from how complete a station is overall. The function below returns the start and end of every interruption longer than a threshold you choose.

Expect an empty result on this window: all three stations record continuously once they start. An empty result is still a result. It tells you that the incomplete station's missing data is one contiguous block at the beginning, rather than the receiver dropping in and out repeatedly. Those are different faults and they call for different responses.

In [ ]:
GAP_THRESHOLD = dt.timedelta(minutes=1)


def find_gaps(frame: pl.DataFrame, station_igs: str, threshold: dt.timedelta) -> pl.DataFrame:
    """Return every interior gap at one station that exceeds `threshold`."""
    epochs = (
        frame.filter(pl.col("igs") == station_igs)
             .select("timestamp")
             .unique()
             .sort("timestamp")
    )
    return (
        epochs.with_columns(pl.col("timestamp").diff().alias("gap_length"))
              .filter(pl.col("gap_length") > threshold)
              .with_columns((pl.col("timestamp") - pl.col("gap_length")).alias("gap_start"))
              .rename({"timestamp": "gap_end"})
              .select(["gap_start", "gap_end", "gap_length"])
    )


for station_igs in df["igs"].unique().sort():
    gaps = find_gaps(df, station_igs, GAP_THRESHOLD)
    print(f"--- {station_igs}: {len(gaps)} gap(s) longer than {GAP_THRESHOLD} ---")
    if len(gaps):
        print(gaps.head(10))

> **Note:** This finds interior gaps only, and that limitation is the whole reason to run it alongside Section 3.2. A station that started recording late, or stopped early, produces no
> difference to detect, because there are no epochs on the missing side to difference against. PCHL is exactly that case: it is missing roughly three quarters of the window and this function reports nothing at all for it. Coverage tells you how much is missing. Gap detection tells you how the data is distributed in time. Both are needed

### 3.5 Per-satellite availability

Coverage can also be uneven within a station. A satellite that is tracked for far fewer epochs than its neighbours may have been obstructed from that site, or may have had its own problems.

We restrict to a single observation code so that each row counts once per satellite per epoch.

In [ ]:
# Raw epoch counts are not comparable across stations here, because the three stations do not
# cover the same amount of wall-clock time. Normalising by each station's own total fixes that.
station_epochs = df.group_by("igs").agg(pl.col("timestamp").n_unique().alias("station_epochs"))

sat_availability = (
    df.filter(pl.col("obs_code") == OBS_CODES[0])
      .group_by(["igs", "satellite"])
      .agg(pl.col("timestamp").n_unique().alias("n_epochs"))
      .join(station_epochs, on="igs")
      .with_columns(
          (pl.col("n_epochs") / pl.col("station_epochs") * 100).round(1).alias("pct_of_window")
      )
      .sort(["igs", "satellite"])
)

# One row per satellite, one column per station: the share of that station's own recorded
# epochs during which the satellite was tracked.
sat_availability.pivot(on="igs", index="satellite", values="pct_of_window").sort("satellite")

**What to look for:** Each figure is the percentage of that station's own recorded epochs during which the satellite was tracked, so the three columns are directly comparable even though
one station covers far less wall-clock time than the others.

Read the table two ways: *down a column* to spot a satellite tracked unusually little at one station, and *across a row* to see whether that satellite is under-represented everywhere.
Section 7 returns to this distinction, because the two cases have very different causes.

> **Check:** Without the normalisation, PCHL's raw counts come out at roughly a third of CLGO's for every satellite, reflecting its shorter recording window rather than anything about the
> satellites themselves. Raw counts are not comparable across stations with unequal coverage.

## 4. Preparing the Data

### 4.1 Accounting for missing values

In [ ]:
df.null_count()

In [ ]:
(
    df.group_by("obs_code")
      .agg(
          pl.len().alias("n_rows"),
          pl.col("snr").null_count().alias("snr_nulls"),
          pl.col("phase").null_count().alias("phase_nulls"),
      )
      .sort("obs_code")
)

> **Note: not every null means the same thing.** Three distinct situations are easy to conflate, and should not be treated alike.
>
> * **A null in a measurement column**, such as `phase` on a row where `snr` is present, means the receiver observed the signal but did not produce that particular observable at that epoch. This is a per-signal tracking outcome, not a fault in the data.
> * **A column that is null throughout** means *nothing to report*.
> * **A missing row** is different in kind. If a satellite was below the horizon there is no row at all, and therefore no null to count. `null_count()` is blind to this entirely.
> 
> Counting nulls tells you about the first two. Only comparing against expected epochs, as in Section 3, tells you about the third.

**What to look for:** Compare the two signals rather than the totals. In this window `1C` carries tens of thousands of `phase` nulls while `2W` carries fewer than a hundred, despite the two having similar row counts. If you plan to use carrier phase, this tells you which signal actually delivers it here.

The cause is not established in this notebook. It would require looking at receiver and firmware behavior, which is outside its scope.

### 4.2 Aggregating into time bins

**What it does:** Collapses the raw 15-second observations into summary statistics over fixed time windows, one per station, satellite, and signal.

**Why it matters:** A million-row dataframe is awkward to plot and slow to reason about, and at 15-second resolution most of what you see is noise rather than structure. Binning to a few
minutes preserves the shape of each satellite pass while cutting the volume by an order of magnitude. Keeping `min`, `max`, and `n_obs` alongside the mean means the aggregation does not hide the variability it is smoothing over.

**Expected output:** The same data reduced from roughly a million rows to a few tens of thousands.

In [ ]:
binned = (
    df.with_columns(
          # Truncate each timestamp down to the start of its bin
          pl.col("timestamp").dt.truncate(TIME_BIN).alias("time_bin"),
          # Composite label so each satellite/signal combination plots as its own series
          pl.concat_str(["system", "satellite", "obs_code"], separator="-").alias("system_sat_obs"),
      )
      .group_by(["igs", "time_bin", "system_sat_obs", "satellite", "obs_code"])
      .agg(
          pl.col("snr").mean().alias("mean_snr"),
          pl.col("snr").min().alias("min_snr"),
          pl.col("snr").max().alias("max_snr"),
          pl.len().alias("n_obs"),
      )
      .sort(["igs", "time_bin"])
)

print(f"{len(df):,} raw rows  ->  {len(binned):,} binned rows")
binned.head()

> **Check:** `n_obs` should be close to 20 for a 5-minute bin at 15-second sampling. Bins with markedly fewer observations sit at the start or end of a satellite pass, or straddle a gap. Polars skips nulls when computing `mean`, so a bin's `mean_snr` is the average of the values that exist rather than null.

## 5. Flagging Outliers

Two quite different things get called outliers in SNR data, and it is worth separating them.

* **Implausible values.** A signal-to-noise density at or below zero, or implausibly high, is not a weak signal. It is a broken number. These are bad data.
* **Low but real values.** Weak readings are correct measurements of an unfavourable situation. They are noisy, but they are not wrong. The next section looks at what drives them here.

> **The thresholds below are author-chosen defaults, not standards.** `SNR_LOW = 30` and `SNR_MAX_PLAUSIBLE = 65` are starting points for this notebook, not published limits. Treat them as parameters to set for your own analysis and check them against your project's conventions before relying on them. 

### Why we flag rather than delete

Low SNR observations are often filtered out, but they are not useless. GNSS reflectometry uses exactly these low-elevation data, because that is where the signal reflected off the ground
interferes with the direct signal strongly enough to measure.

A `snr_quality` column marks them without removing them, so a later workflow can decide for itself which rows it needs.

In [ ]:
SNR_MIN_PLAUSIBLE = 0.0    # dB-Hz, at or below this is not a real measurement
SNR_MAX_PLAUSIBLE = 65.0   # dB-Hz, above this is not physically expected
SNR_LOW           = 30.0   # dB-Hz, below this is weak but real

flagged = df.with_columns(
    pl.when(pl.col("snr").is_null())
      .then(pl.lit("missing"))
      .when(
          (pl.col("snr") <= SNR_MIN_PLAUSIBLE) | (pl.col("snr") > SNR_MAX_PLAUSIBLE)
      )
      .then(pl.lit("implausible"))
      .when(pl.col("snr") < SNR_LOW)
      .then(pl.lit("low"))
      .otherwise(pl.lit("ok"))
      .alias("snr_quality")
)

(
    flagged.group_by("snr_quality")
           .agg(pl.len().alias("n"))
           .with_columns((pl.col("n") / pl.col("n").sum() * 100).round(2).alias("pct"))
           .sort("n", descending=True)
)

In [ ]:
# The same breakdown per station, since signal environment is a property of the site.
(
    flagged.group_by(["igs", "snr_quality"])
           .agg(pl.len().alias("n"))
           .sort(["igs", "snr_quality"])
           .pivot(on="snr_quality", index="igs", values="n")
)

> **Check:** `ok` should account for the large majority of rows and `low` for a minority. On this window the implausible and missing categories never occur, because every snr value is present and inside the plausible range. Note what that does to the pivoted table: pivot creates one column per value it finds, so the result has low and ok and nothing else. A window containing implausible readings would produce a third column.

**What to look for:** The per-station shares come out close to one another, near a fifth of observations in each case. Whatever drives the `low` flag is affecting all three sites to much the same degree, which does not fit a site-specific cause such as local obstruction. The next cell splits the flag a different way.

### 5.1 Interrogating the flag

Before using `snr_quality`, it is worth checking what it is actually responding to. Splitting it by station showed little variation, so the cell below splits it by signal instead.

In [ ]:
(
    flagged.group_by(["obs_code", "snr_quality"])
           .agg(pl.len().alias("n"))
           .sort(["obs_code", "snr_quality"])
           .pivot(on="snr_quality", index="obs_code", values="n")
)

**What to look for:** The `low` flag is not distributed evenly between the two signals. `1C` and `2W` are recorded from the same satellites, at the same epochs, by the same receivers, so a large imbalance here cannot be explained by satellite elevation or by sky obstruction. Those act on both signals at once.

Notebook 1 describes what separates them: `1C` is the open L1 C/A signal, while `2W` is L2 P(Y) recovered by semi-codeless Z-tracking. They are acquired by different methods, and the table above indicates the two do not deliver comparable signal strength.

A single threshold applied to both signals mostly separates `1C` from `2W`. It says little about sky conditions or station health. Two options are reasonable:

* Set a threshold per observation code, so `low` means weak for that signal rather than weak compared with L1 C/A.
* Keep one threshold and always report results split by signal, so the difference stays visible.

## 6. Visualizing Observations

The plots below are drawn from data already aggregated in Polars, which keeps them responsive even though the underlying dataframe has a million rows.

### 6.1 Satellite arcs

The plot below shows mean SNR over time for a handful of satellites at a single station, on a single day.

**What to look for:** Each satellite traces an arc. SNR climbs as the satellite rises above the horizon, peaks near its closest approach to the receiver, and falls away as it sets. Arcs begin and end abruptly because the satellite crosses the horizon. Departures from that smooth shape are the interesting part: a sudden drop in the middle of an arc suggests a temporary obstruction, while a persistently ragged arc suggests multipath from a reflective surface near the antenna.

> **Note:** `DEMO_STATION` takes the alphabetically first station, which here is one with complete coverage. That is the right choice for seeing clean arcs, but it means the incomplete station never appears in this plot. Set `DEMO_STATION = "PCHL00USA"` and re-run the cell to see what a partial record looks like in the same view.

In [ ]:
DEMO_STATION = df["igs"].unique().sort().to_list()[0]   # first station alphabetically
DEMO_SATS    = [5, 7, 11, 13]

DEMO_START   = dt.datetime(2025, 7, 23, tzinfo=dt.timezone.utc)
DEMO_END     = dt.datetime(2025, 7, 24, tzinfo=dt.timezone.utc)

arcs = binned.filter(
    (pl.col("igs") == DEMO_STATION)
    & (pl.col("obs_code") == OBS_CODES[0])
    & (pl.col("satellite").is_in(DEMO_SATS))
    & (pl.col("time_bin") >= DEMO_START)
    & (pl.col("time_bin") < DEMO_END)
)

arcs.plot.point(x="time_bin", y="mean_snr", color="system_sat_obs").properties(
    width=800,
    height=300,
    title=f"{DEMO_STATION}: mean SNR by satellite, {DEMO_START:%Y-%m-%d}",
)

### 6.2 Data availability heatmap

The heatmap below puts satellites on the vertical axis and time on the horizontal, with colour showing how many observations arrived in each hour. It is the visual counterpart to the coverage table from Section 3.

**What to look for:** Colored stripes slanting across the plot are individual satellite passes, which is the normal pattern. Blank regions show where data is absent. A blank vertical band across all satellites means the receiver was down. A blank horizontal band means one satellite was never tracked. A large blank block at one station and not the others is a station outage.

In [ ]:
availability = (
    df.filter(pl.col("obs_code") == OBS_CODES[0])
      .with_columns(pl.col("timestamp").dt.truncate("1h").alias("hour"))
      .group_by(["igs", "hour", "satellite"])
      .agg(pl.len().alias("n_obs"))
)


def availability_chart(frame: pl.DataFrame, station_igs: str) -> alt.Chart:
    """One availability heatmap for a single station."""
    return (
        alt.Chart(frame.filter(pl.col("igs") == station_igs))
           .mark_rect()
           .encode(
               alt.X("hour:T", title="Time (UTC)"),
               alt.Y("satellite:O", title="Satellite"),
               alt.Color("n_obs:Q", title="Obs / hour", scale=alt.Scale(scheme="viridis")),
           )
           .properties(width=700, height=260, title=f"{station_igs}: observation availability")
    )


alt.vconcat(
    *[availability_chart(availability, s) for s in df["igs"].unique().sort()]
)

## 7. Comparing Across Stations and Satellites

A single SNR value is hard to judge on its own. Comparing stations against each other, and satellites against each other, gives you a reference for what is normal in this dataset.

### 7.1 Signal strength distribution by station

In [ ]:
snr_stats = (
    flagged.filter(
        (pl.col("obs_code") == OBS_CODES[0]) & (pl.col("snr_quality") != "implausible")
    )
    .group_by("igs")
    .agg(
        pl.col("snr").mean().round(2).alias("mean"),
        pl.col("snr").median().alias("median"),
        pl.col("snr").std().round(2).alias("std"),
        pl.col("snr").quantile(0.05).alias("p05"),
        pl.col("snr").quantile(0.95).alias("p95"),
    )
    .sort("igs")
)
snr_stats

In [ ]:
# Pre-bin into 1 dB-Hz buckets in Polars so the chart stays light.
histogram = (
    flagged.filter(
        (pl.col("obs_code") == OBS_CODES[0])
        & (pl.col("snr_quality") != "implausible")
        & pl.col("snr").is_not_null()
    )
    .with_columns(pl.col("snr").floor().alias("snr_bin"))
    .group_by(["igs", "snr_bin"])
    .agg(pl.len().alias("n"))
    .sort(["igs", "snr_bin"])
)

histogram.plot.line(x="snr_bin", y="n", color="igs").properties(
    width=800, height=300, title=f"SNR distribution by station ({OBS_CODES[0]})"
)

**What to look for:** On this window the three stations agree closely on center: the means land within a few tenths of a dB-Hz of one another. The differences are in spread instead, so compare `std` and `p05` in the table above rather than `mean`.

Center and spread indicate different things:

* A curve shifted left or right points at the station as a whole: a different antenna, a cable problem, a generally obstructed sky view.
* A curve with the same peak but a fatter low tail points at part of the sky only, which is what partial obstruction looks like.
* Similar center and similar spread means the sites are behaving comparably.

Be careful comparing a station with much shorter coverage against the others. A record spanning a quarter of the window samples a different set of satellite geometries, and some of its difference in spread may come from that rather than from the site.

### 7.2 Per-satellite mean SNR across stations

In [ ]:
sat_comparison = (
    flagged.filter(
        (pl.col("obs_code") == OBS_CODES[0]) & (pl.col("snr_quality") != "implausible")
    )
    .group_by(["igs", "satellite"])
    .agg(
        pl.col("snr").mean().round(2).alias("mean_snr"),
        pl.len().alias("n_obs"),
    )
    .sort(["satellite", "igs"])
)

sat_comparison.plot.point(x="satellite", y="mean_snr", color="igs").properties(
    width=800, height=300, title="Mean SNR per satellite, compared across stations"
)

**What to look for:** This plot separates the two explanations for a weak satellite.

* A satellite that is low at one station but normal at the others points at that station. Something about the site's sky view or hardware disadvantages that part of the sky.
* A satellite that is low at every station points at the satellite itself. Aging hardware and transmitter problems affect all receivers alike.

Satellites with very few observations will have noisy means, so read `mean_snr` alongside `n_obs` rather than on its own.

## 8. Exploration Exercises

Now that you have completed the core workflow, try modifying the parameters below to explore how the results change.

**Try these modifications:**

1. **Change the bin width.** Set `TIME_BIN` in the Configuration section to `"1m"`, then `"30m"`, and re-run Sections 4 and 6.1. At what point does binning stop reducing noise and start erasing the shape of the satellite arcs?

2. **Set thresholds per signal.** Section 5.1 showed that one global `SNR_LOW` mostly separates the two signals rather than separating good conditions from bad. Rewrite the `snr_quality` expression so the threshold depends on `obs_code`, picking a value for each signal from its own distribution in Section 7.1. Does the per-station picture change once the signal effect is taken out?

3. **Move the quality threshold.** Set `SNR_LOW` to 25, then to 35, and re-run Section 5. What fraction of observations moves between `low` and `ok`? What does the sensitivity of that fraction tell you about publishing a fixed threshold as though it were a physical constant?

4. **Find a worse station.** Replace one entry in `STATIONS` with another station, or swap the list for `network_name="PERM:Alaska"` in the Section 2 request to pull the whole network at once. Which station has the largest share of `low` observations, and does its availability heatmap suggest why?

In [ ]:
# Exploration cell - use this space to experiment

## 9. Troubleshooting & Support

### Common Issues

| Error | Likely cause | Fix |
|---|---|---|
| Sampling interval comes back as `0s` | `timestamp` was differenced across all rows, but many rows share one epoch | Reduce to distinct epochs for a single station first: `.select("timestamp").unique().sort("timestamp")` |
| Dataframe has 0 rows | No data for that station, session, or window | Widen `START` / `END`, confirm the 4-character station IDs, or request `network_name` instead |
| `ColumnNotFoundError: phase` | The column was not requested | Add it to `FIELDS` and re-fetch. Server-side field selection means unrequested columns are absent, not null |
| Kernel dies or restarts during the fetch | Request too large for a 4 GB server | Narrow `OBS_CODES`, `FIELDS`, or the time window, or use a query plan as in Notebook 1 Section 6 |
| `mean_snr` is null for some bins | Every `snr` value in that bin was null | Expected at pass edges. Filter on `n_obs` if you need well-populated bins only |

### Further Resources

* [EarthScope SDK - GNSS Observations tutorial](https://docs.earthscope.org/sdk/gnss-obs-tutorial)
* [EarthScope SDK - GNSS Satellite Ephemeris Positions tutorial](https://docs.earthscope.org/sdk/gnss-eph-pos-tutorial)
* [GeoLab Documentation](https://docs.earthscope.org/geolab)
* [Polars User Guide](https://docs.pola.rs/)
* [Altair](https://altair-viz.github.io/)